In [ ]:
%pip -q install tensorflow fastapi uvicorn python-multipart pillow requests

In [ ]:
import os
import io
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import display, Image

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

print("Flattened training shape:", x_train.shape)
print("Flattened testing shape:", x_test.shape)
print("Pixel range:", x_train.min(), "to", x_train.max())

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
], name="MNIST_Digit_Classifier")

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start_time = time.time()

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

training_time = time.time() - start_time
print(f"Training time: {training_time:.2f} seconds")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="Training accuracy")
axes[0].plot(history.history["val_accuracy"], label="Validation accuracy")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history["loss"], label="Training loss")
axes[1].plot(history.history["val_loss"], label="Validation loss")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
pred_probabilities = model.predict(x_test[:10], verbose=0)
predicted_labels = np.argmax(pred_probabilities, axis=1)

plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    color = "green" if predicted_labels[i] == y_test[i] else "red"
    plt.title(f"Pred: {predicted_labels[i]}\nTrue: {y_test[i]}", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
MODEL_PATH = "mnist_digit_model.keras"
model.save(MODEL_PATH)
print("Model saved:", os.path.abspath(MODEL_PATH))

In [ ]:
%%writefile app.py
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from PIL import Image, ImageOps
import numpy as np
import tensorflow as tf
import io
import os

MODEL_PATH = os.environ.get("MODEL_PATH", "mnist_digit_model.keras")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found: {MODEL_PATH}. Run the training notebook first."
    )

model = tf.keras.models.load_model(MODEL_PATH)
app = FastAPI(
    title="MNIST Digit Classification API",
    description="Predicts a handwritten digit from an uploaded image.",
    version="1.0.0"
)

@app.get("/")
def home():
    return {
        "message": "MNIST Digit Classification API is running",
        "docs": "/docs",
        "predict_endpoint": "POST /predict"
    }

@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": True}

def preprocess_image(image_bytes: bytes):
    try:
        image = Image.open(io.BytesIO(image_bytes)).convert("L")
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid image file.")

    # Resize to MNIST input dimensions.
    image = image.resize((28, 28), Image.Resampling.LANCZOS)
    array = np.asarray(image, dtype=np.float32)

    # MNIST samples are light digits on a dark background.
    # Invert automatically when the border is predominantly bright.
    border = np.concatenate([
        array[0, :], array[-1, :], array[:, 0], array[:, -1]
    ])
    if border.mean() > 127:
        array = 255.0 - array

    array = array / 255.0
    return array.reshape(1, 784)

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    if file.content_type not in (
        "image/png", "image/jpeg", "image/jpg", "image/bmp", "image/webp"
    ):
        raise HTTPException(
            status_code=415,
            detail="Upload a PNG, JPG, BMP, or WEBP image."
        )

    image_bytes = await file.read()
    if not image_bytes:
        raise HTTPException(status_code=400, detail="Uploaded file is empty.")
    if len(image_bytes) > 5 * 1024 * 1024:
        raise HTTPException(status_code=413, detail="Maximum image size is 5 MB.")

    input_array = preprocess_image(image_bytes)
    probabilities = model.predict(input_array, verbose=0)[0]
    predicted_digit = int(np.argmax(probabilities))

    return JSONResponse({
        "predicted_digit": predicted_digit,
        "confidence": float(probabilities[predicted_digit]),
        "probabilities": {
            str(i): float(probabilities[i]) for i in range(10)
        }
    })

In [ ]:
# Test the exact image preprocessing used by the API on a sample MNIST image.
from PIL import Image

sample = (x_test[0].reshape(28, 28) * 255).astype("uint8")
sample_image = Image.fromarray(sample)
plt.imshow(sample_image, cmap="gray")
plt.title(f"Sample MNIST digit: {y_test[0]}")
plt.axis("off")
plt.show()

# Direct model prediction (same normalized flattened format).
sample_prediction = model.predict(x_test[0:1], verbose=0)[0]
print("Predicted digit:", int(np.argmax(sample_prediction)))
print("True digit:", int(y_test[0]))

## 15. Run the REST API

### Option A: Run locally (Jupyter / VS Code / terminal)

Keep `app.py` and `mnist_digit_model.keras` in the same folder, then run:

```bash
uvicorn app:app --host 0.0.0.0 --port 8000 --reload
```

Open the interactive Swagger documentation:

`http://127.0.0.1:8000/docs`

Use **POST /predict**, choose an image file, and click **Execute**.

### Option B: Run from Google Colab

Start Uvicorn in a background process:

```python
!uvicorn app:app --host 0.0.0.0 --port 8000 &
```

Colab does not automatically expose localhost publicly. To call it from outside the runtime, configure a secure tunnel (for example, ngrok) and use its generated public URL. Do not expose an unauthenticated API publicly for production use.

In [ ]:
# Run this after starting the API server in another terminal/runtime cell.
# Set API_URL to the reachable server address.
import requests

API_URL = "http://127.0.0.1:8000/predict"

# Create a PNG sample from the first MNIST test image.
sample_path = "mnist_sample.png"
Image.fromarray((x_test[0].reshape(28, 28) * 255).astype("uint8")).save(sample_path)

try:
    with open(sample_path, "rb") as f:
        response = requests.post(
            API_URL,
            files={"file": ("mnist_sample.png", f, "image/png")},
            timeout=30
        )
    print("HTTP status:", response.status_code)
    print(response.json())
except requests.exceptions.RequestException as e:
    print("Could not reach API. Start Uvicorn first.")
    print("Details:", e)